# Mini-Project 2: Uganda Tax Engine

## Sources and assumptions

All rates were checked on 12–13 October 2026. The source of each PAYE schedule is also stored in the code (`BandSchedule.source`).

| Rule | Value used | Source |
|---|---|---|
| Resident PAYE, FY 2025/26 | nil to 235,000; 10%, 20%, 30%; +10% above 10,000,000 | URA PAYE rates page (https://ura.go.ug/?p=16453) |
| Resident PAYE, from 1 July 2026 | nil to 335,000; 20%, 25% (410,001–485,000), 30%; +10% above 10,000,000 | Income Tax (Amendment) Act 2026, commenced 1 July 2026 |
| Non-resident PAYE | 10% to 335,000; 20%; 30%; +10% above 10,000,000 | URA PAYE rates page |
| NSSF | 5% employee, 10% employer, on gross pay | brief baseline; payroll guides |
| Local Service Tax | configurable annual amount, in 4 instalments (July–October), deducted before PAYE | KCCA LST guide |
| VAT | 18% standard, 0% zero-rated, exempt outside VAT | URA VAT page (https://ura.go.ug/?p=38797) |
| Corporate income tax | 30% of chargeable income | professional tax guides (URA source to confirm) |
| Withholding tax | 6% resident fees and designated-agent supplies above UGX 1,000,000; 15% interest, dividends and non-resident payments | URA withholding tax page |
| Rental tax | individuals 12% above UGX 2,820,000; companies 30% after expenses capped at 50% | URA rental income tax page |

**Assumptions and limits:** dates outside FY 2025/26–2026/27 raise `NoScheduleForDateError` rather than guessing; NSSF excludes allowances; LST bands are not hardcoded; no VAT apportionment for mixed supplies; no loss carry-forward; special withholding cases (treaties, listed dividends, 2026 entertainer rules) are out of scope. The rental threshold may have changed under the 2026 Act and should be re-checked.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [2]:
from datetime import date
from src.tax_engine import PAYE, Money, NoScheduleForDateError, TaxContext

paye = PAYE()
for amount in [335_000, 335_001, 410_000, 485_000, 10_000_000, 10_000_001]:
    result = paye.calculate(TaxContext(on=date(2026, 10, 1), gross_pay=Money(amount)))
    print(f"{amount:>12,}  ->  {result.tax}   ({result.note})")

print(paye.calculate(TaxContext(on=date(2026, 3, 1), gross_pay=Money(10_000_000))).tax,
      "(FY 2025/26 resident at 10,000,000)")
print(paye.calculate(TaxContext(on=date(2026, 10, 1), gross_pay=Money(1_000_000),
                                resident=False)).tax, "(non-resident at 1,000,000)")
try:
    paye.calculate(TaxContext(on=date(2028, 1, 1), gross_pay=Money(500_000)))
except NoScheduleForDateError as error:
    print("NoScheduleForDateError:", error)

     335,000  ->  UGX 0   (Resident, FY 2026/27)
     335,001  ->  UGX 0   (Resident, FY 2026/27)
     410,000  ->  UGX 15,000   (Resident, FY 2026/27)
     485,000  ->  UGX 33,750   (Resident, FY 2026/27)
  10,000,000  ->  UGX 2,888,250   (Resident, FY 2026/27)
  10,000,001  ->  UGX 2,888,250   (Resident, FY 2026/27)
UGX 2,902,000 (FY 2025/26 resident at 10,000,000)
UGX 225,500 (non-resident at 1,000,000)
NoScheduleForDateError: No resident PAYE schedule covers 2028-01-01.


In [3]:
from src.tax_engine import PayrollPipeline

payroll = PayrollPipeline(lst_annual=Money(30_000))   # example annual LST
for month in (date(2026, 10, 1), date(2026, 11, 1)):
    slip = payroll.run(TaxContext(on=month, gross_pay=Money(1_000_000)))
    print(month, "| NSSF", slip.nssf_employee.tax, "| LST", slip.lst.tax,
          "| PAYE", slip.paye.tax, "| net", slip.net_pay,
          "| employer NSSF", slip.nssf_employer.tax)

2026-10-01 | NSSF UGX 50,000 | LST UGX 7,500 | PAYE UGX 186,000 | net UGX 756,500 | employer NSSF UGX 100,000
2026-11-01 | NSSF UGX 50,000 | LST UGX 0 | PAYE UGX 188,250 | net UGX 761,750 | employer NSSF UGX 100,000


In [4]:
from src.tax_engine import CorporateIncomeTax, Invoice, SupplyType, VAT

invoices = (
    Invoice(Money(1_000_000), SupplyType.STANDARD, is_sale=True),
    Invoice(Money(500_000), SupplyType.ZERO_RATED, is_sale=True),
    Invoice(Money(200_000), SupplyType.EXEMPT, is_sale=True),
    Invoice(Money(400_000), SupplyType.STANDARD, is_sale=False),
    Invoice(Money(100_000), SupplyType.STANDARD, is_sale=False, for_exempt_use=True),
    Invoice(Money(50_000), SupplyType.EXEMPT, is_sale=False),
)
vat = VAT()
ctx = TaxContext(on=date(2026, 10, 1), invoices=invoices)
print("Output VAT:", vat.output_vat(invoices), "| Input VAT:", vat.input_vat(invoices),
      "| Net payable:", vat.calculate(ctx).tax)

cit = CorporateIncomeTax()
print("CIT profit:", cit.calculate(TaxContext(on=date(2026, 10, 1),
      revenue=Money(50_000_000), deductions=Money(35_000_000))).tax)
print("CIT loss:  ", cit.calculate(TaxContext(on=date(2026, 10, 1),
      revenue=Money(10_000_000), deductions=Money(12_000_000))).tax)

Output VAT: UGX 180,000 | Input VAT: UGX 72,000 | Net payable: UGX 108,000
CIT profit: UGX 4,500,000
CIT loss:   UGX 0


In [5]:
from src.tax_engine import PaymentType, RentalIncomeTax, WithholdingTax

on = date(2026, 10, 1)
fees = WithholdingTax(PaymentType.PROFESSIONAL_FEES)
goods = WithholdingTax(PaymentType.GOODS_AND_SERVICES)
print("Fees 2,000,000, resident:     ", fees.calculate(TaxContext(on=on, revenue=Money(2_000_000))).tax)
print("Fees 2,000,000, non-resident: ", fees.calculate(TaxContext(on=on, revenue=Money(2_000_000), resident=False)).tax)
print("Goods 800,000 (below 1m):     ", goods.calculate(TaxContext(on=on, revenue=Money(800_000))).tax)
print("Goods 1,500,000:              ", goods.calculate(TaxContext(on=on, revenue=Money(1_500_000))).tax)

print("Rental, individual 6,000,000: ", RentalIncomeTax().calculate(TaxContext(on=on, revenue=Money(6_000_000))).tax)
print("Rental, company 30m, exp 20m: ", RentalIncomeTax(individual=False).calculate(
      TaxContext(on=on, revenue=Money(30_000_000), deductions=Money(20_000_000))).tax)

Fees 2,000,000, resident:      UGX 120,000
Fees 2,000,000, non-resident:  UGX 300,000
Goods 800,000 (below 1m):      UGX 0
Goods 1,500,000:               UGX 90,000
Rental, individual 6,000,000:  UGX 381,600
Rental, company 30m, exp 20m:  UGX 4,500,000


## 5. Report discussion

In [6]:
from decimal import Decimal

print("0.1 + 0.2 with float:   ", 0.1 + 0.2)
print("0.1 + 0.2 with Decimal: ", Decimal("0.1") + Decimal("0.2"))
print("Ten 0.1 amounts sum to 1 (float):  ", sum([0.1] * 10) == 1.0)
print("Ten 0.1 amounts sum to 1 (Decimal):", sum([Decimal("0.1")] * 10) == Decimal("1"))

0.1 + 0.2 with float:    0.30000000000000004
0.1 + 0.2 with Decimal:  0.3
Ten 0.1 amounts sum to 1 (float):   True
Ten 0.1 amounts sum to 1 (Decimal): True


5.1 Why float is unsafe for tax calculations

Floating-point numbers are stored in binary (base 2), which cannot represent most decimal fractions exactly. As the cell above shows, 0.1 + 0.2 gives 0.30000000000000004 with float, while Decimal, which uses base-10 arithmetic, gives exactly 0.3. Adding ten amounts of 0.1 with float does not even equal 1.

In tax calculations, these tiny errors lead to financial discrepancies. A result that should be exactly on a band boundary can land slightly above or below it, and rounding can then move the final tax by a shilling. Across thousands of payslips or invoices, the differences add up, and the engine's figures would no longer match URA's.

The Money class prevents this by encapsulating a Decimal. It rejects floats outright with a TypeError, keeps arithmetic exact, and applies one explicit rounding rule (ROUND_HALF_UP to whole shillings) once, at the end of each calculation.

5.2 Where inheritance is justified, and where it is avoided

Inheritance is used where classes share a core interface. PAYE, VAT, CorporateIncomeTax, NSSF, LocalServiceTax, WithholdingTax and RentalIncomeTax are all different calculations, but every one of them takes a TaxContext and returns a TaxResult. The abstract Tax base class enforces this with calculate(self, ctx), so any tax can be used in the same way, for example inside the payroll pipeline.

Inheritance is deliberately avoided where classes would differ only in their data. Resident and non-resident PAYE work in exactly the same way and differ only in their bands and rates, so they are BandSchedule objects held by PAYE (composition over inheritance), not subclasses. The same applies to NSSF: the employee and employer contributions are one class reused with two rates (5% and 10%), so the logic exists in one place. Withholding rates are likewise a table keyed by payment type and residency. Adding a new tax year or rate means adding data, not writing a new class.

5.3 Extension A: withholding tax and rental income tax

The withholding tax module covers standard commercial payments: professional fees, designated-agent supplies of goods and services above UGX 1,000,000, interest and dividends, at URA's resident and non-resident rates. Out of scope are exempt payments and exemption certificates, salaries (already covered by PAYE), treaty and listed-company rates, and the new 2026 withholding rules.

The rental income tax module applies URA's two methods: 12% of individual gross rent above the UGX 2,820,000 threshold, and 30% for companies after expenses capped at 50%. The tests confirm correct threshold logic: the individual case reproduces URA's own worked example exactly (UGX 381,600), and a designated-agent payment of exactly UGX 1,000,000 correctly attracts no withholding tax.